# 生産計画サンプル

## ライブラリ

In [ ]:
import random
from itertools import combinations
from datetime import datetime, timedelta
from collections import defaultdict

from ortools.sat.python import cp_model
import pandas as pd
import plotly.express as px

## 入力データ

In [ ]:
# --------------------------
# パラメータ（要件）
# --------------------------
NUM_LOTS = 101
NUM_LINES = 12
NUM_MATERIALS = 30
SLOT_MINUTES = 10
LOT_DURATION_MIN = 30
SLOTS_PER_LOT = LOT_DURATION_MIN // SLOT_MINUTES  # =3
WORK_START = datetime(2025, 9, 14, 8, 0)  # 可視化用の基準日（任意）
WORK_END = datetime(2025, 9, 14, 17, 0)
TOTAL_SLOTS = int(((WORK_END - WORK_START).seconds // 60) // SLOT_MINUTES)  # 54
LAST_START_SLOT = TOTAL_SLOTS - SLOTS_PER_LOT  # 54 - 3 = 51

random.seed(0)

## 最適化

In [ ]:

# --------------------------
# 在庫 (inventory)
# - 各ロットは2材料を消費する想定 -> 合計必要数 = 2 * NUM_LOTS
# - ここでは例として在庫をランダムに生成するが、実データがあれば置き換えてください
# --------------------------
total_need = 2 * NUM_LOTS
# Create random inventories but ensure sum >= total_need
inventory = [random.randint(0, 10) for _ in range(NUM_MATERIALS)]
s = sum(inventory)
if s < total_need:
    # top-up randomly until sum >= total_need
    i = 0
    while s < total_need:
        inventory[i % NUM_MATERIALS] += 1
        s += 1
        i += 1

print("Inventory (sample first 10):", inventory[:10], " Sum:", sum(inventory))

# --------------------------
# 仮の損失行列 W を用意（実データに置き換えてください）
# W[a][b] = materials a,b の組合せで発生する「材料ロス」（非負整数）
# 対称行列とする
# --------------------------
W = [[0]*NUM_MATERIALS for _ in range(NUM_MATERIALS)]
for i in range(NUM_MATERIALS):
    for j in range(i+1, NUM_MATERIALS):
        # 仮定: ランダムな損失 0..10
        val = random.randint(0, 10)
        W[i][j] = val
        W[j][i] = val

# --------------------------
# Stage 1: 材料割当（在庫制約を追加）
# --------------------------
model1 = cp_model.CpModel()

# x[m,l] = 1 if material m assigned to lot l
x = {}
for m in range(NUM_MATERIALS):
    for l in range(NUM_LOTS):
        x[(m, l)] = model1.NewBoolVar(f"x_m{m}_l{l}")

# Each lot exactly 2 materials
for l in range(NUM_LOTS):
    model1.Add(sum(x[(m, l)] for m in range(NUM_MATERIALS)) == 2)

# Inventory constraint: for each material m, assigned count <= inventory[m]
for m in range(NUM_MATERIALS):
    model1.Add(sum(x[(m, l)] for l in range(NUM_LOTS)) <= inventory[m])

# y[a,b,l] for unordered pairs a<b to linearize objective
pair_indices = list(combinations(range(NUM_MATERIALS), 2))
y = {}
for (a, b) in pair_indices:
    for l in range(NUM_LOTS):
        y[(a, b, l)] = model1.NewBoolVar(f"y_{a}_{b}_l{l}")
        model1.Add(y[(a, b, l)] <= x[(a, l)])
        model1.Add(y[(a, b, l)] <= x[(b, l)])
        model1.Add(y[(a, b, l)] >= x[(a, l)] + x[(b, l)] - 1)

# Objective: minimize total material loss
obj_terms = []
for (a, b) in pair_indices:
    w = W[a][b]
    if w != 0:
        for l in range(NUM_LOTS):
            obj_terms.append(w * y[(a, b, l)])
model1.Minimize(sum(obj_terms))

solver1 = cp_model.CpSolver()
solver1.parameters.max_time_in_seconds = 60.0
solver1.parameters.num_search_workers = 8
print("Solving Stage 1 (material assignment with inventory)...")
res1 = solver1.Solve(model1)
if res1 not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    raise RuntimeError("Stage1: No feasible solution found")

# Extract results
lot_materials = []
lot_type = []
for l in range(NUM_LOTS):
    mats = [m for m in range(NUM_MATERIALS) if solver1.Value(x[(m, l)]) == 1]
    if len(mats) != 2:
        raise RuntimeError(f"Stage1: lot {l} doesn't have 2 materials (got {mats})")
    a, b = sorted(mats)
    lot_materials.append((a, b))
    lot_type.append((a, b))

print("Stage1 done. Example assigned pairs (first 10 lots):")
for i in range(min(10, NUM_LOTS)):
    print(f" Lot {i}: materials {lot_materials[i]}, loss W={W[lot_materials[i][0]][lot_materials[i][1]]}")

# Map types
type_to_idx = {}
idx_to_type = {}
lots_of_type = defaultdict(list)
type_counter = 0
for l, t in enumerate(lot_type):
    if t not in type_to_idx:
        type_to_idx[t] = type_counter
        idx_to_type[type_counter] = t
        type_counter += 1
    tidx = type_to_idx[t]
    lots_of_type[tidx].append(l)
NUM_TYPES = type_counter
print(f"Number of distinct lot types used: {NUM_TYPES}")

In [ ]:
# --------------------------
# Stage 2: スケジューリング（隣接材料一致を出来るだけ満たす）
# - 在庫は Stage1 で満たしているので、Stage2 は主に時間割とライン上の隣接最小化
# --------------------------
model2 = cp_model.CpModel()

# start[l, line, s] bool
start = {}
for l in range(NUM_LOTS):
    for line in range(NUM_LINES):
        for s in range(LAST_START_SLOT + 1):
            start[(l, line, s)] = model2.NewBoolVar(f"start_l{l}_line{line}_s{s}")

# Each lot exactly once
for l in range(NUM_LOTS):
    model2.Add(sum(start[(l, line, s)]
                   for line in range(NUM_LINES)
                   for s in range(LAST_START_SLOT + 1)) == 1)

# No overlap on a line
for line in range(NUM_LINES):
    for tau in range(TOTAL_SLOTS):
        occupying = []
        s_min = max(0, tau - SLOTS_PER_LOT + 1)
        s_max = min(LAST_START_SLOT, tau)
        for l in range(NUM_LOTS):
            for s in range(s_min, s_max + 1):
                occupying.append(start[(l, line, s)])
        model2.Add(sum(occupying) <= 1)

# Forbid immediate start at s_next if types differ (ensures 10min switch)
for line in range(NUM_LINES):
    for s in range(LAST_START_SLOT + 1):
        s_next = s + SLOTS_PER_LOT
        if s_next <= LAST_START_SLOT:
            # For each pair of types p != q, ensure they don't both appear as starts at (line,s) and (line,s_next)
            # Efficient implementation: for each type p, sum_p_s = sum(start(l in type p, line, s)), sum_q_snext similar
            # then for p and q != p, sum_p_s + sum_q_snext <= 1  (disallow different-type immediate adjacency)
            # This is equivalent to: if there's p at s and q at s_next and p!=q -> forbidden
            # Compute sums per type
            sum_p_s = {}
            sum_p_snext = {}
            for p in range(NUM_TYPES):
                sum_p_s[p] = sum(start[(l, line, s)] for l in lots_of_type[p])
                sum_p_snext[p] = sum(start[(l, line, s_next)] for l in lots_of_type[p])
            # Now forbid different type adjacency
            for p in range(NUM_TYPES):
                for q in range(NUM_TYPES):
                    if p == q:
                        continue
                    model2.Add(sum_p_s[p] + sum_p_snext[q] <= 1)

# Now build variables to measure adjacent starts and same-type adjacency:
# For each line and s where s_next exists:
#   sum_s = sum starts at (line,s)
#   sum_snext = sum starts at (line,s_next)
#   total_both is 1 iff both sums are 1
#   For each type p, same_p is 1 iff there is a start of type p at s AND a start of same type p at s_next
#   diff_var >= total_both - sum_p same_p  (approx: diff_var = total_both - number_of_same_type_pairs)
diff_vars = []
for line in range(NUM_LINES):
    for s in range(LAST_START_SLOT + 1):
        s_next = s + SLOTS_PER_LOT
        if s_next <= LAST_START_SLOT:
            # sums
            sum_s = sum(start[(l, line, s)] for l in range(NUM_LOTS))
            sum_snext = sum(start[(l, line, s_next)] for l in range(NUM_LOTS))
            # total_both bool
            total_both = model2.NewBoolVar(f"both_line{line}_s{s}")
            # total_both <= sum_s and <= sum_snext
            model2.Add(total_both <= sum_s)
            model2.Add(total_both <= sum_snext)
            # if sum_s + sum_snext >= 2 then total_both can be 1; enforce lower bound:
            model2.Add(total_both >= sum_s + sum_snext - 1)

            # same_p for each type
            same_ps = []
            for p in range(NUM_TYPES):
                sum_p_s = sum(start[(l, line, s)] for l in lots_of_type[p]) if lots_of_type[p] else 0
                sum_p_snext = sum(start[(l, line, s_next)] for l in lots_of_type[p]) if lots_of_type[p] else 0
                if isinstance(sum_p_s, int) and sum_p_s == 0:
                    # no lots of this type exist, skip
                    continue
                sp = model2.NewBoolVar(f"same_line{line}_s{s}_type{p}")
                # sp <= sum_p_s, sp <= sum_p_snext
                model2.Add(sp <= sum_p_s)
                model2.Add(sp <= sum_p_snext)
                # sp >= sum_p_s + sum_p_snext -1
                model2.Add(sp >= sum_p_s + sum_p_snext - 1)
                same_ps.append(sp)

            # diff_var measures number of differing-type adjacencies at this (line,s)
            # We want diff_var = total_both - sum(same_ps) (>=0)
            tot_same = sum(same_ps) if same_ps else 0  # linear expression
            # diff_var as integer (0..1)
            diff = model2.NewIntVar(0, len(lots_of_type), f"diff_line{line}_s{s}")
            # diff >= total_both - tot_same
            # CP-SAT requires linearization: diff >= total_both - tot_same  -> rearrange: diff + tot_same >= total_both
            model2.Add(diff + tot_same >= total_both)
            # diff <= total_both  (can't exceed total_both)
            model2.Add(diff <= total_both)
            # diff >= 0 already enforced by var definition
            diff_vars.append(diff)

# Objective: minimize sum of diff_vars (i.e., minimize adjacent differing-type starts)
# This encourages consecutive lots on same line to have same type (thus satisfying "隣接でなるべく同じ材料" 要件)
model2.Minimize(sum(diff_vars))

solver2 = cp_model.CpSolver()
solver2.parameters.max_time_in_seconds = 120.0  # 調整可
solver2.parameters.num_search_workers = 8
print("Solving Stage 2 (scheduling with adjacency preference)...")
res2 = solver2.Solve(model2)
if res2 not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    raise RuntimeError("Stage2: No feasible solution found")

## 可視化

In [ ]:
# Extract schedule: for each lot, find (line, start)
lot_assignment = {}
for l in range(NUM_LOTS):
    found = False
    for line in range(NUM_LINES):
        for s in range(LAST_START_SLOT + 1):
            if solver2.Value(start[(l, line, s)]) == 1:
                lot_assignment[l] = {"line": line, "start_slot": s}
                found = True
                break
        if found:
            break
    if not found:
        raise RuntimeError(f"lot {l} has no assignment (shouldn't happen)")

# Build DataFrame for Gantt chart
rows = []
for l in range(NUM_LOTS):
    info = lot_assignment[l]
    sslot = info["start_slot"]
    start_dt = WORK_START + timedelta(minutes=sslot * SLOT_MINUTES)
    end_dt = start_dt + timedelta(minutes=LOT_DURATION_MIN)
    type_pair = lot_type[l]
    rows.append({
        "Lot": f"Lot_{l}",
        "Start": start_dt,
        "Finish": end_dt,
        "Line": f"Line_{info['line']}",
        "Materials": f"{type_pair[0]} & {type_pair[1]}",
        "TypeIdx": type_to_idx[type_pair]
    })

df = pd.DataFrame(rows)

# Sort for nicer plotting
df = df.sort_values(by=["Line", "Start"])

# Plotly Gantt (using timeline)
fig = px.timeline(df, x_start="Start", x_end="Finish", y="Line", color="Materials",
                  hover_data=["Lot", "Materials"])
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Production Gantt Chart: Lots assigned to lines and times",
                  xaxis_title="Time", yaxis_title="Line", height=700)

# Show or save
# If running in notebook, use fig.show()
# Otherwise, save to html
output_html = "gantt_schedule.html"
fig.show()
fig.write_html(output_html)
print(f"Schedule written to {output_html}")

# Also print a small textual summary for first few lots
print("\nFirst 10 lot assignments:")
for l in range(min(10, NUM_LOTS)):
    a, b = lot_type[l]
    asg = lot_assignment[l]
    start_dt = WORK_START + timedelta(minutes=asg["start_slot"]*SLOT_MINUTES)
    print(f" Lot {l}: type ({a},{b}), Line {asg['line']}, Start {start_dt.time()} -- duration {LOT_DURATION_MIN}min")

# Save schedule as CSV for convenience
df.to_csv("schedule_summary.csv", index=False)
print("CSV saved to schedule_summary.csv")